# BKMeeting AI Hub Option 1 NPU Pilots

This notebook adapts the minimal `On_device_Ai.ipynb` example into a repo-specific Qualcomm AI Hub workflow for BKMeeting.

Scope of this notebook:

- stay fully in `python-model-test`
- do not touch Android packaging
- prove compile, profile, and inference on Qualcomm AI Hub first
- split each pilot into:
  - `prepare + compile only`
  - `resolve existing compiled target + run + compare`
- run two pilots:
  - Zipformer encoder-first
  - VPCD model-session-first


## Environment Notes

Before running this notebook, make sure the current environment can already execute the local `python-model-test` bundle helpers.

Minimum practical dependencies:

- `qai-hub`
- `torch`
- `torchaudio`
- `numpy`
- local editable install of this repo if needed

The Zipformer pilot uses the existing repo feature-extraction path, so `torchaudio` must be available.


In [1]:
!pip install qai-hub "qai-hub[torch]"


In [2]:
from pathlib import Path
import os
import sys

import qai_hub as hub

sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_pilots import resolve_qai_hub_api_token

API_TOKEN = resolve_qai_hub_api_token(repo_root=Path.cwd())

if not API_TOKEN:
    print("Set QAI_HUB_API_TOKEN in .env or your shell environment before running Qualcomm AI Hub configuration.")
else:
    os.environ["QAI_HUB_API_TOKEN"] = API_TOKEN
    available_devices = hub.get_devices()
    print("Loaded QAI_HUB_API_TOKEN from .env or shell environment.")
    print("AI Hub device count:", len(available_devices))
    print("AI Hub first devices:")
    for device in available_devices[:5]:
        print(device)


D:\Anaconda\envs\speech2text\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded QAI_HUB_API_TOKEN from .env or shell environment.
AI Hub device count: 80
AI Hub first devices:
Device(name='Google Pixel 3 (Family)', os='10', attributes=['os:android', 'framework:tflite', 'framework:onnx', 'abi:aarch64-android', 'vendor:google', 'format:phone', 'chipset:qualcomm-snapdragon-845', 'chipset:sdm845', 'hexagon:v65', 'soc-model:1'])
Device(name='Google Pixel 3', os='10', attributes=['os:android', 'framework:tflite', 'framework:onnx', 'abi:aarch64-android', 'vendor:google', 'format:phone', 'chipset:qualcomm-snapdragon-845', 'chipset:sdm845', 'hexagon:v65', 'soc-model:1'])
Device(name='Google Pixel 3a', os='10', attributes=['os:android', 'framework:tflite', 'framework:onnx', 'abi:aarch64-android', 'vendor:google', 'format:phone', 'chipset:qualcomm-snapdragon-670', 'chipset:sdm670', 'hexagon:v65', 'soc-model:6'])
Device(name='Google Pixel 3 XL', os='10', attributes=['os:android', 'framework:tflite', 'framework:onnx', 'abi:aarch64-android', 'vendor:google', 'format:phon

In [3]:
import sys
from pathlib import Path

import onnxruntime as ort
import qai_hub as hub

from model_bundle.fixtures import read_jsonl
sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_hybrid_pipeline import (
    run_vpcd_hybrid_evaluation,
    run_zipformer_hybrid_evaluation,
)
from tools.aihub_option1_pilots import (
    build_compile_options,
    build_job_options,
    build_option1_runtime_config,
    build_vpcd_input_specs,
    build_vpcd_single_step_calibration_entries,
    build_vpcd_single_step_inputs,
    build_zipformer_encoder_inference_entries,
    build_zipformer_encoder_input_specs,
    compare_output_tensors,
    coerce_inputs_for_compiled_model,
    prepare_vpcd_option1_source_model,
    prepare_zipformer_encoder_option1_source_model,
    resolve_target_model_id,
    resolve_vpcd_fp32_source_model_path,
    resolve_vpcd_pilot_source,
    resolve_zipformer_encoder_pilot_source,
    summarize_vpcd_step_logits,
    write_compile_run_record,
    write_live_run_record,
    write_prepared_artifact_record,
    wrap_single_inference_inputs,
)


In [4]:
DEVICE_NAME = "Samsung Galaxy S24 (Family)"
QAIRT_VERSION = None
RUN_LABEL = "latest"

# Use one stable RUN_LABEL per compiled artifact set, for example: "s24-main" or "debug-2026-05-12".
# If you want to reuse a previous compile without recompiling, keep the same RUN_LABEL so the notebook can
# read build/aihub/records/<pilot>/compile-run-<RUN_LABEL>.json automatically.

# Optional: paste an existing compiled target model id here to bypass compile-record lookup entirely.
# Leave these as None if you want the Resolve step to load the model id from compile-run-<RUN_LABEL>.json.
ZIPFORMER_TARGET_MODEL_ID = None
VPCD_TARGET_MODEL_ID = None

# If a compile record already exists for the current RUN_LABEL, the compile cells now skip automatically.
# They only submit compile jobs when no explicit target model id is set and no compile-run record exists yet.
AUTO_SKIP_COMPILE_IF_RECORD_EXISTS = True

# Phase 3 hybrid loops reuse the same compiled target ids and RUN_LABEL from Phase 2.
ZIPFORMER_HYBRID_MAX_SAMPLES = 2
VPCD_HYBRID_MAX_SAMPLES = 4

RUNTIME_CONFIG = build_option1_runtime_config(
    device_name=DEVICE_NAME,
    qairt_version=QAIRT_VERSION,
    repo_root=Path.cwd(),
)
job_options = build_job_options(
    compute_unit=RUNTIME_CONFIG.compute_unit,
    qairt_version=RUNTIME_CONFIG.qairt_version,
)

print("device:", RUNTIME_CONFIG.device_name)
print("qairt_version:", RUNTIME_CONFIG.qairt_version)
print("artifact_root:", RUNTIME_CONFIG.artifact_root)
print("record_root:", RUNTIME_CONFIG.record_root)
print("job_options:", job_options)
print("run_label:", RUN_LABEL)
print("zipformer reuse target model id:", ZIPFORMER_TARGET_MODEL_ID)
print("vpcd reuse target model id:", VPCD_TARGET_MODEL_ID)
print("auto skip compile if record exists:", AUTO_SKIP_COMPILE_IF_RECORD_EXISTS)
print("zipformer hybrid max samples:", ZIPFORMER_HYBRID_MAX_SAMPLES)
print("vpcd hybrid max samples:", VPCD_HYBRID_MAX_SAMPLES)


device: Samsung Galaxy S24 (Family)
qairt_version: None
artifact_root: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub
record_root: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records
job_options: --compute_unit npu
run_label: latest
zipformer reuse target model id: None
vpcd reuse target model id: None
auto skip compile if record exists: True
zipformer hybrid max samples: 2
vpcd hybrid max samples: 4


## How To Use This Notebook

This notebook supports two common workflows, and each pilot now has both a **Phase 2 tensor diagnostic path** and a **Phase 3 hybrid e2e path**.

### Workflow A: Compile From Scratch, Then Run And Compare

Use this when you do **not** already have a compiled target model for the current pilot.

1. Run the setup cells from the top through the config cell.
2. Keep `ZIPFORMER_TARGET_MODEL_ID = None` and `VPCD_TARGET_MODEL_ID = None` unless you want to force a specific target id manually.
3. Choose a `RUN_LABEL` for this compile, for example `latest`, `s24-main`, or `debug-2026-05-12`.
4. For each pilot you want to test, run these sections in order:
   - `Prepare`
   - `Compile Only`
   - `Resolve Existing Compiled Target`
   - `Run And Compare Against The Compiled Target`
   - `Output Inspection (Intermediate Diagnostic Only)`
   - `Hybrid E2E Run`
   - `Final Compare Against Expected Outputs` or `Final Compare Against Gold Samples`
5. After compile succeeds, the notebook writes `compile-run-<RUN_LABEL>.json` under `build/aihub/records/<pilot>/`.
6. On later days, you can reuse that same compile by keeping the same `RUN_LABEL` and skipping the compile cell.

### Workflow B: Skip Compile, Reuse An Existing Compiled Target, Then Run And Compare

Use this when compile already succeeded earlier and you only want to rerun inference and correctness checks.

You have two ways to reuse an existing compiled target:

1. **Recommended:** keep `RUN_LABEL` the same as the earlier compile and leave `*_TARGET_MODEL_ID = None`.
   - The `Resolve Existing Compiled Target` cell will load the target model id from `compile-run-<RUN_LABEL>.json`.
2. **Manual override:** paste a known target model id into `ZIPFORMER_TARGET_MODEL_ID` or `VPCD_TARGET_MODEL_ID`.
   - This bypasses record lookup and uses that exact compiled target directly.

When reusing a previous compile, run only these sections for the pilot:

- `Prepare`
- `Resolve Existing Compiled Target`
- `Run And Compare Against The Compiled Target`
- `Output Inspection (Intermediate Diagnostic Only)`
- `Hybrid E2E Run`
- `Final Compare Against Expected Outputs` or `Final Compare Against Gold Samples`

You can safely skip the `Compile Only` section in this workflow.


## Pilot 1: Zipformer Encoder-First

This pilot targets the first ASR slice that BKMeeting wants to offload first: the encoder graph.

The current local helper now prepares a dedicated AI Hub upload artifact from the fixed-shape encoder source.

- base source: fixed-shape encoder ONNX
- upload artifact: ORT-optimized + symbolic-shape-prepared + HTP bool-slice rewrite
- local fixtures: current Zipformer bundle sample manifest
- current verified lane: direct `submit_compile_job(...)` on the prepared source model


In [5]:
zipformer_pilot_name = "zipformer_encoder_option1"
zipformer_source = resolve_zipformer_encoder_pilot_source(RUNTIME_CONFIG.repo_root)
zipformer_source_model_path = prepare_zipformer_encoder_option1_source_model(
    zipformer_source,
    output_path=RUNTIME_CONFIG.pilot_artifact_dir(zipformer_pilot_name) / "encoder.aihub.option1.onnx",
)
zipformer_input_specs = build_zipformer_encoder_input_specs(zipformer_source)
zipformer_compile_options = build_compile_options(
    qairt_version=RUNTIME_CONFIG.qairt_version,
    input_specs=zipformer_input_specs,
)
zipformer_raw_inference_inputs = build_zipformer_encoder_inference_entries(zipformer_source)
zipformer_inference_inputs = coerce_inputs_for_compiled_model(
    zipformer_raw_inference_inputs,
    input_specs=zipformer_input_specs,
)
zipformer_prepared_record_path = write_prepared_artifact_record(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    source_model_path=zipformer_source.source_model_path,
    prepared_model_path=zipformer_source_model_path,
    input_specs=zipformer_input_specs,
    compile_options=zipformer_compile_options,
    run_label=RUN_LABEL,
)

print("zipformer base source model:", zipformer_source.source_model_path)
print("zipformer prepared upload model:", zipformer_source_model_path)
print("zipformer bundle manifest:", zipformer_source.bundle_manifest_path)
print("zipformer input specs:", zipformer_input_specs)
print("zipformer compile options:", zipformer_compile_options)
print("zipformer prepared record:", zipformer_prepared_record_path)
print({name: [value.shape for value in values] for name, values in zipformer_inference_inputs.items()})


Unable to determine if floor(If_597_o0__d0/2) + 501 <= If_597_o0__d0, treat as equal
Cannot determine if floor(If_597_o0__d0/2) - 500 < 0
Unable to determine if floor(If_1168_o0__d0/2) + 251 <= If_1168_o0__d0, treat as equal
Cannot determine if floor(If_1168_o0__d0/2) - 250 < 0
Unable to determine if floor(If_1739_o0__d0/2) + 126 <= If_1739_o0__d0, treat as equal
Cannot determine if floor(If_1739_o0__d0/2) - 125 < 0
Unable to determine if floor(If_2308_o0__d0/2) + 251 <= If_2308_o0__d0, treat as equal
Cannot determine if floor(If_2308_o0__d0/2) - 250 < 0
Unable to determine if floor(If_2877_o0__d0/2) + 501 <= If_2877_o0__d0, treat as equal
Cannot determine if floor(If_2877_o0__d0/2) - 500 < 0


zipformer base source model: D:\DS-AI\BKMeeting-Research\python-model-test\build\quantize\zipformer\qnn_u16u8\fixed_shapes\encoder.fixed.onnx
zipformer prepared upload model: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\zipformer_encoder_option1\encoder.aihub.option1.onnx
zipformer bundle manifest: D:\DS-AI\BKMeeting-Research\python-model-test\build\model_bundle\zipformer\qnn_u16u8\bundle_manifest.json
zipformer input specs: {'x': ((1, 2009, 80), 'float32'), 'x_lens': ((1,), 'int64')}
zipformer compile options: --target_runtime precompiled_qnn_onnx --truncate_64bit_io
zipformer prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\prepared-artifact-latest.json
{'x': [(1, 2009, 80)], 'x_lens': [(1,)]}


### Zipformer Compile Only

Use this section only when you need to create a **new compiled target model** for Zipformer.

Run this section when:

- this is your first time testing Zipformer on the selected cloud device
- you changed the prepared source model or compile options
- you want a fresh compiled artifact under a new `RUN_LABEL`

After this cell succeeds, save or remember at least one of these:

- `RUN_LABEL`
- `zipformer target model id`
- the record file `build/aihub/records/zipformer_encoder_option1/compile-run-<RUN_LABEL>.json`

If you only want to rerun inference and compare outputs, do **not** rerun this section. Jump to `Resolve Existing Compiled Target` instead.


In [6]:
zipformer_compile_record_target = RUNTIME_CONFIG.pilot_record_dir(zipformer_pilot_name) / f"compile-run-{RUN_LABEL}.json"
zipformer_should_compile = not (
    AUTO_SKIP_COMPILE_IF_RECORD_EXISTS
    and ZIPFORMER_TARGET_MODEL_ID is None
    and zipformer_compile_record_target.exists()
)
zipformer_compile_job = None
zipformer_compiled_target_model = None
zipformer_compile_record_path = zipformer_compile_record_target

if ZIPFORMER_TARGET_MODEL_ID is not None:
    print("Skipping Zipformer compile because ZIPFORMER_TARGET_MODEL_ID is set.")
elif zipformer_should_compile:
    zipformer_compile_job = hub.submit_compile_job(
        model=zipformer_source_model_path,
        device=hub.Device(RUNTIME_CONFIG.device_name),
        input_specs=zipformer_input_specs,
        options=zipformer_compile_options,
        name="bkmeeting-zipformer-encoder-precompiled-qnn-onnx",
    )
    zipformer_compiled_target_model = zipformer_compile_job.get_target_model()
    zipformer_compile_record_path = write_compile_run_record(
        pilot_name=zipformer_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        compile_options=zipformer_compile_options,
        compile_job=zipformer_compile_job,
        target_model=zipformer_compiled_target_model,
        run_label=RUN_LABEL,
    )

    print("zipformer compile job:", zipformer_compile_job.url)
    print("zipformer target model id:", zipformer_compiled_target_model.model_id)
    print("zipformer target model url:", zipformer_compiled_target_model.url)
    print("zipformer compile record:", zipformer_compile_record_path)
else:
    print("Skipping Zipformer compile because compile record already exists:", zipformer_compile_record_target)


Skipping Zipformer compile because compile record already exists: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\compile-run-latest.json


### Resolve Existing Compiled Target

This section decides **which compiled Zipformer target model** will be used for profile, inference, and comparison.

It works in two modes:

1. `ZIPFORMER_TARGET_MODEL_ID = None`
   - the notebook reads `build/aihub/records/zipformer_encoder_option1/compile-run-<RUN_LABEL>.json`
   - use this when you want to reuse a previous compile by label
2. `ZIPFORMER_TARGET_MODEL_ID = "..."`
   - the notebook skips record lookup and uses that exact model id directly
   - use this when you copied a target model id from an earlier notebook run or AI Hub page

If this cell fails with a missing record error, it usually means one of these:

- you never ran `Compile Only` for this `RUN_LABEL`
- you changed `RUN_LABEL` and the matching `compile-run-<RUN_LABEL>.json` does not exist yet
- you should paste a known `ZIPFORMER_TARGET_MODEL_ID` manually


In [7]:
zipformer_target_model_id = resolve_target_model_id(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    explicit_target_model_id=ZIPFORMER_TARGET_MODEL_ID,
    run_label=RUN_LABEL,
)
zipformer_target_model = hub.get_model(zipformer_target_model_id)

print("zipformer resolved target model id:", zipformer_target_model_id)
print("zipformer target model url:", zipformer_target_model.url)


zipformer resolved target model id: mnzljr3zm
zipformer target model url: https://workbench.aihub.qualcomm.com/models/mnzljr3zm/


### Run And Compare Against The Compiled Target

This is the **fast rerun loop** for Zipformer.

Use this section when:

- compile already exists and you want to rerun on the cloud NPU device
- you want fresh profile/inference jobs without paying compile time again
- you want to compare cloud output against the local CPU baseline again

This section does three things:

1. profile the already-compiled target model on the selected cloud device
2. run inference on the same compiled target model
3. write a fresh `live-run-<RUN_LABEL>.json` record and leave `zipformer_output` ready for the inspection cell

After this cell finishes, run the `Zipformer Output Inspection` cell right below it.


In [8]:
zipformer_profile_job = hub.submit_profile_job(
    model=zipformer_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    options=job_options,
    name="bkmeeting-zipformer-encoder-profile-npu",
)

zipformer_profile = zipformer_profile_job.download_profile()
zipformer_inference_job = hub.submit_inference_job(
    model=zipformer_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    inputs=zipformer_inference_inputs,
    options=job_options,
    name="bkmeeting-zipformer-encoder-inference-npu",
)
zipformer_output = zipformer_inference_job.download_output_data()
zipformer_live_record_path = write_live_run_record(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=zipformer_compile_options,
    job_options=job_options,
    compile_job=zipformer_compile_job if "zipformer_compile_job" in globals() else {"status": "reused-target-model"},
    profile_job=zipformer_profile_job,
    inference_job=zipformer_inference_job,
    output_tensors=zipformer_output,
    run_label=RUN_LABEL,
)

print("zipformer profile job:", zipformer_profile_job.url)
print("zipformer inference job:", zipformer_inference_job.url)
print("zipformer live record:", zipformer_live_record_path)
print("zipformer output tensors:", {name: [value.shape for value in values] for name, values in zipformer_output.items()})


Scheduled profile job (jgjklxr15) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgjklxr15/

Waiting for profile job (jgjklxr15) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


Uploading dataset: 264kB [00:01, 229kB/s]                    <?, ?B/s]


Scheduled inference job (jg994v7vg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jg994v7vg/

Waiting for inference job (jg994v7vg) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpjdqgqcq5.h5: 100%|██████████| 529k/529k [00:00<00:00, 981kB/s] 

zipformer profile job: https://workbench.aihub.qualcomm.com/jobs/jgjklxr15/
zipformer inference job: https://workbench.aihub.qualcomm.com/jobs/jg994v7vg/
zipformer live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\live-run-latest.json
zipformer output tensors: {'output_0': [(1, 501, 512)], 'output_1': [(1,)]}


## Zipformer Output Inspection (Intermediate Diagnostic Only)

This section checks encoder tensors only.
Use it to sanity-check that the compiled target stays close to the local CPU ONNX baseline before running the slower hybrid transcript path.

Do **not** treat this as the final correctness gate.
The final transcript comparison happens in the `Zipformer Hybrid E2E Run` and `Zipformer Final Compare Against Expected Outputs` sections right below.


In [9]:
zipformer_cpu_inputs = {name: values[0] for name, values in zipformer_raw_inference_inputs.items()}
zipformer_cpu_session = ort.InferenceSession(
    zipformer_source.source_model_path.as_posix(),
    providers=["CPUExecutionProvider"],
)
zipformer_cpu_output_arrays = zipformer_cpu_session.run(None, zipformer_cpu_inputs)
zipformer_cpu_output = {f"output_{index}": [value] for index, value in enumerate(zipformer_cpu_output_arrays)}
zipformer_output_comparison = compare_output_tensors(
    zipformer_cpu_output,
    zipformer_output,
    atol=1e-3,
    rtol=1e-3,
)
zipformer_expected_outputs = read_jsonl(zipformer_source.bundle_manifest_path.parent / "expected_outputs.jsonl")

print("zipformer reference transcript:", zipformer_expected_outputs[0]["text"] if zipformer_expected_outputs else "n/a")
print("zipformer encoder_out_lens (cloud):", zipformer_output["output_1"][0].tolist())
print("zipformer encoder frame preview (cloud):")
print(zipformer_output["output_0"][0][0, :2, :8])
zipformer_output_comparison


zipformer reference transcript: ▁CHÀO▁CÁC▁BẠN▁HÔM▁NAY▁CHÚNG▁TA▁CÙNG▁NHAU▁ĐẾN▁VỚI▁BÀI▁HỌC▁DEP▁LEARNING▁PHẦN▁SỐ▁MƯỜI▁BA▁ĐÁNG▁LÝ▁BÀI▁NÀY▁ĐÃ▁HỌC▁TỪ▁NGÀY▁HAI▁MƯƠI▁MỐT▁THÁNG▁MƯỜI▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁NĂM▁NHƯNG▁VÌ▁NGHỈ▁TẾT▁CHÚNG▁TA▁GIỜ▁LỊCH▁ĐẾN▁NGÀY▁HAI▁MƯƠI▁HAI▁THÁNG▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁SÁU
zipformer encoder_out_lens (cloud): [216]
zipformer encoder frame preview (cloud):
[[-0.47973636  0.69042975  0.09777833  0.189209    0.1859131   0.0300293
   0.5585938   0.05712891]
 [-0.4306641   0.7509766   0.18078615 -0.06140137  0.22790529  0.18579103
   0.5595704  -0.29687503]]


{'output_0': {'reference_dtype': 'float32',
  'candidate_dtype': 'float32',
  'reference_shape': [1, 501, 512],
  'candidate_shape': [1, 501, 512],
  'shape_match': True,
  'allclose': False,
  'max_abs_diff': 0.6959928832948208,
  'mean_abs_diff': 0.009590339702975922},
 'output_1': {'reference_dtype': 'int64',
  'candidate_dtype': 'int32',
  'reference_shape': [1],
  'candidate_shape': [1],
  'shape_match': True,
  'allclose': True,
  'max_abs_diff': 0.0,
  'mean_abs_diff': 0.0}}

### Zipformer Hybrid E2E Run

Run this section only after the compiled target has already been resolved.
This is the first point where the notebook executes the real Phase 3 hybrid pipeline:

1. feature extraction on the host
2. encoder inference on the compiled cloud NPU target
3. greedy decoder and joiner on the host CPU
4. write `hybrid-run-<RUN_LABEL>.json` under `build/aihub/records/zipformer_hybrid_option1/`


In [10]:
zipformer_hybrid_report = run_zipformer_hybrid_evaluation(
    runtime_config=RUNTIME_CONFIG,
    run_label=RUN_LABEL,
    explicit_target_model_id=ZIPFORMER_TARGET_MODEL_ID,
    max_samples=ZIPFORMER_HYBRID_MAX_SAMPLES,
)
zipformer_hybrid_record_path = zipformer_hybrid_report["record_path"]

print("zipformer hybrid target model id:", zipformer_hybrid_report["target_reference"].target_model_id)
print("zipformer hybrid summary:", zipformer_hybrid_report["summary"])
print("zipformer hybrid record:", zipformer_hybrid_record_path)


Uploading dataset: 405kB [00:01, 324kB/s]                            2.60MB/s]


Scheduled inference job (jgzv7ee6p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgzv7ee6p/

Waiting for inference job (jgzv7ee6p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpklabh1dx.h5: 100%|██████████| 603k/603k [00:00<00:00, 1.06MB/s]
Uploading dataset: 562kB [00:00, 1.94MB/s]                   <?, ?B/s]


Scheduled inference job (jgoev06xp) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgoev06xp/

Waiting for inference job (jgoev06xp) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmp71eubmoc.h5: 100%|██████████| 647k/647k [00:00<00:00, 1.11MB/s]


zipformer hybrid target model id: mnzljr3zm
zipformer hybrid summary: {'sample_count': 2, 'comparable_samples': 2, 'matched_samples': 0, 'mismatched_samples': 2, 'mismatch_items': ['sample-1', 'sample-2'], 'comparison_unavailable_samples': 0, 'comparison_unavailable_items': []}
zipformer hybrid record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_hybrid_option1\hybrid-run-latest.json


### Zipformer Final Compare Against Expected Outputs

This is the final correctness gate for Zipformer in this notebook.
Only this section decides whether the evaluated samples match `expected_outputs.jsonl` end to end.


In [11]:
zipformer_hybrid_results = zipformer_hybrid_report["results"]
zipformer_hybrid_comparable = [row for row in zipformer_hybrid_results if row["matches_expected"] is not None]
zipformer_hybrid_mismatches = [row for row in zipformer_hybrid_comparable if not row["matches_expected"]]
zipformer_hybrid_unavailable = [row for row in zipformer_hybrid_results if row["matches_expected"] is None]

print("zipformer final transcript compare:")
for row in zipformer_hybrid_results:
    print(
        {
            "sample_id": row["sample_id"],
            "audio_path": row["audio_path"],
            "text": row["text"],
            "expected_text": row["expected_text"],
            "expected_available": row["expected_available"],
            "matches_expected": row["matches_expected"],
            "cloud_inference_seconds": row["cloud_inference_seconds"],
            "decode_seconds": row["decode_seconds"],
        }
    )

if zipformer_hybrid_unavailable:
    print("zipformer rows without expected transcript fixture:")
    for row in zipformer_hybrid_unavailable:
        print({"sample_id": row["sample_id"], "audio_path": row["audio_path"]})

if zipformer_hybrid_mismatches:
    print("zipformer mismatches:")
    for row in zipformer_hybrid_mismatches:
        print(
            {
                "sample_id": row["sample_id"],
                "audio_path": row["audio_path"],
                "text": row["text"],
                "expected_text": row["expected_text"],
            }
        )
elif zipformer_hybrid_comparable:
    print("zipformer all comparable samples matched expected transcripts.")
else:
    print("zipformer final compare could not run because no expected transcript fixtures were available.")


zipformer final transcript compare:
{'sample_id': 'sample-1', 'audio_path': 'assets/speech/sample-1.mp3', 'text': '▁CHÀO▁CÁC▁BẠN▁HÔM▁NAY▁CHÚNG▁TA▁CÙNG▁NHAU▁ĐẾN▁VỚI▁BÀI▁HỌC▁DEP▁LEARNING▁PHẦN▁SỐ▁MƯỜI▁BA▁ĐÁNG▁LÝ▁BÀI▁NÀY▁ĐÃ▁HỌC▁TỪ▁NGÀY▁HAI▁MƯƠI▁MỐT▁THÁNG▁MƯỜI▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁LĂM▁NHƯNG▁VÌ▁NGHỈ▁TẾT▁CHÚNG▁TA▁GIỜ▁LỊCH▁ĐẾN▁NGÀY▁HAI▁MƯƠI▁HAI▁THÁNG▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁SÁU', 'expected_text': '▁CHÀO▁CÁC▁BẠN▁HÔM▁NAY▁CHÚNG▁TA▁CÙNG▁NHAU▁ĐẾN▁VỚI▁BÀI▁HỌC▁DEP▁LEARNING▁PHẦN▁SỐ▁MƯỜI▁BA▁ĐÁNG▁LÝ▁BÀI▁NÀY▁ĐÃ▁HỌC▁TỪ▁NGÀY▁HAI▁MƯƠI▁MỐT▁THÁNG▁MƯỜI▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁NĂM▁NHƯNG▁VÌ▁NGHỈ▁TẾT▁CHÚNG▁TA▁GIỜ▁LỊCH▁ĐẾN▁NGÀY▁HAI▁MƯƠI▁HAI▁THÁNG▁HAI▁NĂM▁HAI▁NGHÌN▁KHÔNG▁TRĂM▁HAI▁MƯƠI▁SÁU', 'expected_available': True, 'matches_expected': False, 'cloud_inference_seconds': 176.706396, 'decode_seconds': 0.527078}
{'sample_id': 'sample-2', 'audio_path': 'assets/speech/sample-2.wav', 'text': '▁Ê▁HÔM▁NAY▁MỆT▁XỈU▁LUÔN▁Á▁SÁNG▁ĐI▁LÀM▁BỊ▁XẾP▁ASEAN▁TEPROSEC▁CẤP▁RÚT▁DEADLINE▁THÌ▁GẦN

## Pilot 2: VPCD Model-Session-First

This pilot targets the punctuation model session while keeping tokenization on the host side.

Important current caveats:

- prefer the repo FP32 export when available, then freeze it to the fixed bundle shapes before upload
- if the source is still QDQ after preparation, compile it directly as the pragmatic fallback lane
- compiled inference inputs must be coerced from `int64` to `int32` when `--truncate_64bit_io` is present


In [12]:
vpcd_pilot_name = "vpcd_option1"
vpcd_source = resolve_vpcd_pilot_source(RUNTIME_CONFIG.repo_root)
vpcd_original_source_model_path = resolve_vpcd_fp32_source_model_path(vpcd_source) or vpcd_source.model_path
vpcd_prepared_source_model_path, vpcd_is_quantized_source = prepare_vpcd_option1_source_model(
    vpcd_source,
    output_path=RUNTIME_CONFIG.pilot_artifact_dir(vpcd_pilot_name) / "model.option1.onnx",
)
vpcd_input_specs = build_vpcd_input_specs(vpcd_source)
vpcd_compile_options = build_compile_options(
    qairt_version=RUNTIME_CONFIG.qairt_version,
    input_specs=vpcd_input_specs,
)
vpcd_calibration_data = build_vpcd_single_step_calibration_entries(vpcd_source, max_samples=4)
vpcd_single_step_inputs = build_vpcd_single_step_inputs(vpcd_source, sample_index=0)
vpcd_raw_inference_inputs = wrap_single_inference_inputs(vpcd_single_step_inputs)
vpcd_inference_inputs = coerce_inputs_for_compiled_model(
    vpcd_raw_inference_inputs,
    input_specs=vpcd_input_specs,
)
vpcd_prepared_record_path = write_prepared_artifact_record(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    source_model_path=vpcd_original_source_model_path,
    prepared_model_path=vpcd_prepared_source_model_path,
    input_specs=vpcd_input_specs,
    compile_options=vpcd_compile_options,
    run_label=RUN_LABEL,
)

print("vpcd source model:", vpcd_original_source_model_path)
print("vpcd prepared upload model:", vpcd_prepared_source_model_path)
print("vpcd input specs:", vpcd_input_specs)
print("vpcd compile options:", vpcd_compile_options)
print("vpcd quantized source:", vpcd_is_quantized_source)
print("vpcd prepared record:", vpcd_prepared_record_path)
print({name: [value.shape for value in values] for name, values in vpcd_inference_inputs.items()})


vpcd source model: D:\DS-AI\BKMeeting-Research\python-model-test\assets\vietnamese-punc-cap-denorm-v1\onnx\model.fp32.onnx
vpcd prepared upload model: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\vpcd_option1\model.option1.onnx
vpcd input specs: {'input_ids': ((1, 1024), 'int64'), 'attention_mask': ((1, 1024), 'int64'), 'decoder_input_ids': ((1, 128), 'int64'), 'decoder_attention_mask': ((1, 128), 'int64')}
vpcd compile options: --target_runtime precompiled_qnn_onnx --truncate_64bit_io
vpcd quantized source: False
vpcd prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\prepared-artifact-latest.json
{'input_ids': [(1, 1024)], 'attention_mask': [(1, 1024)], 'decoder_input_ids': [(1, 128)], 'decoder_attention_mask': [(1, 128)]}


### VPCD Compile Only

Use this section only when you need to create a **new compiled target model** for VPCD.

Run this section when:

- this is your first time testing VPCD on the selected cloud device
- you changed the prepared source model, quantize step, or compile options
- you want a fresh compiled artifact under a new `RUN_LABEL`

After this cell succeeds, save or remember at least one of these:

- `RUN_LABEL`
- `vpcd target model id`
- the record file `build/aihub/records/vpcd_option1/compile-run-<RUN_LABEL>.json`

If you only want to rerun inference and compare outputs, do **not** rerun this section. Jump to `Resolve Existing Compiled Target` instead.


In [13]:
vpcd_compile_record_target = RUNTIME_CONFIG.pilot_record_dir(vpcd_pilot_name) / f"compile-run-{RUN_LABEL}.json"
vpcd_should_compile = not (
    AUTO_SKIP_COMPILE_IF_RECORD_EXISTS
    and VPCD_TARGET_MODEL_ID is None
    and vpcd_compile_record_target.exists()
)
vpcd_quantize_job = None
vpcd_compile_job = None
vpcd_compiled_target_model = None
vpcd_compile_record_path = vpcd_compile_record_target

if VPCD_TARGET_MODEL_ID is not None:
    print("Skipping VPCD compile because VPCD_TARGET_MODEL_ID is set.")
elif vpcd_should_compile:
    if vpcd_is_quantized_source:
        vpcd_compile_input_model = vpcd_prepared_source_model_path
        print("VPCD source is already QDQ. Compiling directly for the current AI Hub pilot.")
    else:
        vpcd_quantize_job = hub.submit_quantize_job(
            model=vpcd_prepared_source_model_path,
            calibration_data=vpcd_calibration_data,
            name="bkmeeting-vpcd-quantize",
        )
        vpcd_compile_input_model = vpcd_quantize_job.get_target_model()
        print("vpcd quantize job:", vpcd_quantize_job.url)

    vpcd_compile_job = hub.submit_compile_job(
        model=vpcd_compile_input_model,
        device=hub.Device(RUNTIME_CONFIG.device_name),
        input_specs=vpcd_input_specs,
        options=vpcd_compile_options,
        name="bkmeeting-vpcd-precompiled-qnn-onnx",
    )
    vpcd_compiled_target_model = vpcd_compile_job.get_target_model()
    vpcd_compile_record_path = write_compile_run_record(
        pilot_name=vpcd_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        compile_options=vpcd_compile_options,
        compile_job=vpcd_compile_job,
        target_model=vpcd_compiled_target_model,
        run_label=RUN_LABEL,
    )

    print("vpcd compile job:", vpcd_compile_job.url)
    print("vpcd target model id:", vpcd_compiled_target_model.model_id)
    print("vpcd target model url:", vpcd_compiled_target_model.url)
    print("vpcd compile record:", vpcd_compile_record_path)
else:
    print("Skipping VPCD compile because compile record already exists:", vpcd_compile_record_target)


Skipping VPCD compile because compile record already exists: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\compile-run-latest.json


### Resolve Existing Compiled Target

This section decides **which compiled VPCD target model** will be used for profile, inference, and comparison.

It works in two modes:

1. `VPCD_TARGET_MODEL_ID = None`
   - the notebook reads `build/aihub/records/vpcd_option1/compile-run-<RUN_LABEL>.json`
   - use this when you want to reuse a previous compile by label
2. `VPCD_TARGET_MODEL_ID = "..."`
   - the notebook skips record lookup and uses that exact model id directly
   - use this when you copied a target model id from an earlier notebook run or AI Hub page

If this cell fails with a missing record error, it usually means one of these:

- you never ran `Compile Only` for this `RUN_LABEL`
- you changed `RUN_LABEL` and the matching `compile-run-<RUN_LABEL>.json` does not exist yet
- you should paste a known `VPCD_TARGET_MODEL_ID` manually


In [14]:
vpcd_target_model_id = resolve_target_model_id(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    explicit_target_model_id=VPCD_TARGET_MODEL_ID,
    run_label=RUN_LABEL,
)
vpcd_target_model = hub.get_model(vpcd_target_model_id)

print("vpcd resolved target model id:", vpcd_target_model_id)
print("vpcd target model url:", vpcd_target_model.url)


vpcd resolved target model id: mnjeogxym
vpcd target model url: https://workbench.aihub.qualcomm.com/models/mnjeogxym/


### Run And Compare Against The Compiled Target

This is the **fast rerun loop** for VPCD.

Use this section when:

- compile already exists and you want to rerun on the cloud NPU device
- you want fresh profile/inference jobs without paying compile time again
- you want to compare cloud output against the local CPU baseline again

This section does three things:

1. profile the already-compiled target model on the selected cloud device
2. run inference on the same compiled target model
3. write a fresh `live-run-<RUN_LABEL>.json` record and leave `vpcd_output` ready for the inspection cell

After this cell finishes, run the `VPCD Output Inspection` cell right below it.


In [15]:
vpcd_profile_job = hub.submit_profile_job(
    model=vpcd_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    options=job_options,
    name="bkmeeting-vpcd-profile-npu",
)

vpcd_profile = vpcd_profile_job.download_profile()
vpcd_inference_job = hub.submit_inference_job(
    model=vpcd_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    inputs=vpcd_inference_inputs,
    options=job_options,
    name="bkmeeting-vpcd-inference-npu",
)
vpcd_output = vpcd_inference_job.download_output_data()
vpcd_live_record_path = write_live_run_record(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=vpcd_compile_options,
    job_options=job_options,
    compile_job=vpcd_compile_job if "vpcd_compile_job" in globals() else {"status": "reused-target-model"},
    profile_job=vpcd_profile_job,
    inference_job=vpcd_inference_job,
    output_tensors=vpcd_output,
    run_label=RUN_LABEL,
)

print("vpcd profile job:", vpcd_profile_job.url)
print("vpcd inference job:", vpcd_inference_job.url)
print("vpcd live record:", vpcd_live_record_path)
print("vpcd output tensors:", {name: [value.shape for value in values] for name, values in vpcd_output.items()})


Scheduled profile job (j57vdwlv5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j57vdwlv5/

Waiting for profile job (j57vdwlv5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


Uploading dataset: 25.3kB [00:00, 41.2kB/s]                   <?, ?B/s]


Scheduled inference job (jp23m63qg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp23m63qg/

Waiting for inference job (jp23m63qg) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmprs1w3ou6.h5: 100%|██████████| 1.12M/1.12M [00:00<00:00, 1.59MB/s]


vpcd profile job: https://workbench.aihub.qualcomm.com/jobs/j57vdwlv5/
vpcd inference job: https://workbench.aihub.qualcomm.com/jobs/jp23m63qg/
vpcd live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\live-run-latest.json
vpcd output tensors: {'output_0': [(1, 128, 40030)], 'output_1': [(1, 1024, 1024)]}


## VPCD Output Inspection (Intermediate Diagnostic Only)

This section checks one model-step tensor only.
Use it to sanity-check logits and token preference on the compiled target before running the full hybrid decode loop.

Do **not** treat this as the final correctness gate.
The final punctuation comparison happens in the `VPCD Hybrid E2E Run` and `VPCD Final Compare Against Gold Samples` sections right below.


In [16]:
vpcd_cpu_inputs = {name: value for name, value in vpcd_single_step_inputs.items()}
vpcd_cpu_session = ort.InferenceSession(
    vpcd_prepared_source_model_path.as_posix(),
    providers=["CPUExecutionProvider"],
)
vpcd_cpu_output_arrays = vpcd_cpu_session.run(None, vpcd_cpu_inputs)
vpcd_cpu_output = {f"output_{index}": [value] for index, value in enumerate(vpcd_cpu_output_arrays)}
vpcd_output_comparison = compare_output_tensors(
    vpcd_cpu_output,
    vpcd_output,
    atol=1e-2,
    rtol=1e-2,
)
vpcd_golden_sample = read_jsonl(vpcd_source.golden_samples_path)[0]
vpcd_cpu_next_token_summary = summarize_vpcd_step_logits(
    vpcd_cpu_output["output_0"][0],
    vpcd_single_step_inputs["decoder_attention_mask"],
    top_k=5,
)
vpcd_next_token_summary = summarize_vpcd_step_logits(
    vpcd_output["output_0"][0],
    vpcd_single_step_inputs["decoder_attention_mask"],
    top_k=5,
)

print("vpcd raw_text:", vpcd_golden_sample["raw_text"])
print("vpcd expected_output:", vpcd_golden_sample["expected_output"])
print("vpcd active decoder index:", vpcd_next_token_summary["active_index"])
print("vpcd cpu top next-token candidates:")
for item in vpcd_cpu_next_token_summary["top_tokens"]:
    print(item)
print("vpcd cloud top next-token candidates:")
for item in vpcd_next_token_summary["top_tokens"]:
    print(item)
vpcd_output_comparison


vpcd raw_text: hôm nay là buổi nhậm chức của tôi phước thành
vpcd expected_output: Hôm nay là buổi nhậm chức của tôi - Phước Thành.
vpcd active decoder index: 0
vpcd cpu top next-token candidates:
{'token_id': 0, 'score': 72.7752685546875}
{'token_id': 889, 'score': 53.407981872558594}
{'token_id': 581, 'score': 53.317874908447266}
{'token_id': 1191, 'score': 53.01805877685547}
{'token_id': 118, 'score': 52.341957092285156}
vpcd cloud top next-token candidates:
{'token_id': 0, 'score': 49.22596740722656}
{'token_id': 6609, 'score': 49.22596740722656}
{'token_id': 17786, 'score': 49.22596740722656}
{'token_id': 947, 'score': 49.22596740722656}
{'token_id': 889, 'score': 49.22596740722656}


{'output_0': {'reference_dtype': 'float32',
  'candidate_dtype': 'float32',
  'reference_shape': [1, 128, 40030],
  'candidate_shape': [1, 128, 40030],
  'shape_match': True,
  'allclose': False,
  'max_abs_diff': 61.031280517578125,
  'mean_abs_diff': 45.44200608524837},
 'output_1': {'reference_dtype': 'float32',
  'candidate_dtype': 'float32',
  'reference_shape': [1, 1024, 1024],
  'candidate_shape': [1, 1024, 1024],
  'shape_match': True,
  'allclose': False,
  'max_abs_diff': 4.1272929310798645,
  'mean_abs_diff': 0.3763680103623363}}

### VPCD Hybrid E2E Run

Run this section only after the compiled target has already been resolved.
This is the first point where the notebook executes the real Phase 3 hybrid pipeline:

1. tokenizer encode on the host CPU
2. compiled model-step inference on the cloud NPU target
3. host-side decode loop until EOS or max length
4. write `hybrid-run-<RUN_LABEL>.json` under `build/aihub/records/vpcd_hybrid_option1/`


In [17]:
vpcd_hybrid_report = run_vpcd_hybrid_evaluation(
    runtime_config=RUNTIME_CONFIG,
    run_label=RUN_LABEL,
    explicit_target_model_id=VPCD_TARGET_MODEL_ID,
    max_samples=VPCD_HYBRID_MAX_SAMPLES,
)
vpcd_hybrid_record_path = vpcd_hybrid_report["record_path"]

print("vpcd hybrid target model id:", vpcd_hybrid_report["target_reference"].target_model_id)
print("vpcd hybrid summary:", vpcd_hybrid_report["summary"])
print("vpcd hybrid record:", vpcd_hybrid_record_path)


Uploading dataset: 25.3kB [00:00, 40.7kB/s]                   <?, ?B/s]


Scheduled inference job (jp4jwo12p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp4jwo12p/

Waiting for inference job (jp4jwo12p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpl9lygs0j.h5: 100%|██████████| 1.12M/1.12M [00:00<00:00, 1.57MB/s]
Uploading dataset: 25.3kB [00:00, 42.2kB/s]                   <?, ?B/s]


Scheduled inference job (jgoev094p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgoev094p/

Waiting for inference job (jgoev094p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmp6rau9dtr.h5: 100%|██████████| 1.12M/1.12M [00:00<00:00, 1.57MB/s]
Uploading dataset: 25.3kB [00:00, 42.3kB/s]                   <?, ?B/s]


...

Scheduled inference job (jgjklm485) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgjklm485/

Waiting for inference job (jgjklm485) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpsxqm01yd.h5: 100%|██████████| 1.12M/1.12M [00:00<00:00, 1.58MB/s]
Uploading dataset: 25.3kB [00:00, 101kB/s]                    <?, ?B/s]


Scheduled inference job (jp8w7vyxp) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp8w7vyxp/

Waiting for inference job (jp8w7vyxp) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpebbsxgjy.h5: 100%|██████████| 1.12M/1.12M [00:00<00:00, 1.61MB/s]


vpcd hybrid target model id: mnjeogxym
vpcd hybrid summary: {'sample_count': 2, 'comparable_samples': 2, 'matched_samples': 0, 'mismatched_samples': 2, 'mismatch_items': [0, 1], 'comparison_unavailable_samples': 0, 'comparison_unavailable_items': []}
vpcd hybrid record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_hybrid_option1\hybrid-run-latest.json


### VPCD Final Compare Against Gold Samples

This is the final correctness gate for VPCD in this notebook.
Only this section decides whether the evaluated samples match `golden_samples.jsonl` end to end.


In [18]:
vpcd_hybrid_results = vpcd_hybrid_report["results"]
vpcd_hybrid_mismatches = [row for row in vpcd_hybrid_results if not row["matches_expected"]]

print("vpcd final punctuation compare:")
for row in vpcd_hybrid_results:
    print(
        {
            "sample_index": row["sample_index"],
            "raw_text": row["raw_text"],
            "text": row["text"],
            "expected_text": row["expected_text"],
            "matches_expected": row["matches_expected"],
            "decode_steps": row["decode_steps"],
            "generated_ids": row["generated_ids"],
            "golden_input_ids": row["golden_input_ids"],
            "cloud_inference_seconds": row["cloud_inference_seconds"],
            "decode_seconds": row["decode_seconds"],
        }
    )

if vpcd_hybrid_mismatches:
    print("vpcd mismatches:")
    for row in vpcd_hybrid_mismatches:
        print(
            {
                "sample_index": row["sample_index"],
                "raw_text": row["raw_text"],
                "text": row["text"],
                "expected_text": row["expected_text"],
                "generated_ids": row["generated_ids"],
            }
        )
else:
    print("vpcd all evaluated samples matched golden outputs.")


vpcd final punctuation compare:
{'sample_index': 0, 'raw_text': 'hôm nay là buổi nhậm chức của tôi phước thành', 'text': '⁇', 'expected_text': 'Hôm nay là buổi nhậm chức của tôi - Phước Thành.', 'matches_expected': False, 'decode_steps': 3, 'generated_ids': [0, 1, 2], 'golden_input_ids': [0, 799, 177, 9, 847, 559, 2306, 115, 7, 80, 1386, 1338, 58, 2], 'cloud_inference_seconds': 716.382445, 'decode_seconds': 716.389152}
{'sample_index': 1, 'raw_text': 'chào các bạn hôm nay chúng ta cùng nhau đến với bài học deep learning phần số mười ba', 'text': '⁇', 'expected_text': 'Chào các bạn, hôm nay chúng ta cùng nhau đến với bài học Deep Learning phần số 13.', 'matches_expected': False, 'decode_steps': 3, 'generated_ids': [0, 1, 2], 'golden_input_ids': [0, 1740, 10, 144, 799, 177, 248, 336, 120, 383, 30, 15, 635, 71, 19466, 18436, 221, 52, 3125, 712, 2], 'cloud_inference_seconds': 717.915429, 'decode_seconds': 717.92141}
vpcd mismatches:
{'sample_index': 0, 'raw_text': 'hôm nay là buổi nhậm chứ

## After The Notebook Runs

This notebook now leaves behind a deterministic evidence trail for both Phase 2 and Phase 3:

- prepared upload artifact under `build/aihub/<pilot>/`
- prepared artifact record under `build/aihub/records/<pilot>/prepared-artifact-<RUN_LABEL>.json`
- compile-only record under `build/aihub/records/<pilot>/compile-run-<RUN_LABEL>.json`
- live run record under `build/aihub/records/<pilot>/live-run-<RUN_LABEL>.json`
- hybrid e2e record under `build/aihub/records/<pilot>_hybrid_option1/hybrid-run-<RUN_LABEL>.json`
- AI Hub job URLs printed in the execution cells

Recommended habit:

- use one stable `RUN_LABEL` for one compiled artifact set
- when you want to reuse compile later, keep the same `RUN_LABEL` and skip the `Compile Only` cell
- if you already know a target model id, paste it into `*_TARGET_MODEL_ID` and go straight to `Resolve Existing Compiled Target`
- only treat the notebook as correctness-complete after the final transcript or punctuation compare cells finish


In [19]:
print("runtime record root:", RUNTIME_CONFIG.record_root)
print("zipformer prepared record:", zipformer_prepared_record_path)
print("zipformer compile record:", RUNTIME_CONFIG.pilot_record_dir(zipformer_pilot_name) / f"compile-run-{RUN_LABEL}.json")
print("zipformer live record:", RUNTIME_CONFIG.pilot_record_dir(zipformer_pilot_name) / f"live-run-{RUN_LABEL}.json")
print("zipformer hybrid record:", globals().get("zipformer_hybrid_record_path"))
print("vpcd prepared record:", vpcd_prepared_record_path)
print("vpcd compile record:", RUNTIME_CONFIG.pilot_record_dir(vpcd_pilot_name) / f"compile-run-{RUN_LABEL}.json")
print("vpcd live record:", RUNTIME_CONFIG.pilot_record_dir(vpcd_pilot_name) / f"live-run-{RUN_LABEL}.json")
print("vpcd hybrid record:", globals().get("vpcd_hybrid_record_path"))


runtime record root: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records
zipformer prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\prepared-artifact-latest.json
zipformer compile record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\compile-run-latest.json
zipformer live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_encoder_option1\live-run-latest.json
zipformer hybrid record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\zipformer_hybrid_option1\hybrid-run-latest.json
vpcd prepared record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\prepared-artifact-latest.json
vpcd compile record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_option1\compile-run-latest.json
vpcd live record: D:\DS-AI\BKMeeting-Research\python-model-test\build\aihub\records\vpcd_opt